In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
from pathlib import Path
PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))
from data.load_opsd import load_opsd_time_series_15min
plt.style.use("seaborn-v0_8")
DATA_DIR = Path("../data/processed")
DATA_DIR.mkdir(exist_ok=True)
import requests
from io import BytesIO
from zipfile import ZipFile
OPSD_TIME_SERIES_URL = "https://data.open-power-system-data.org/time_series/opsd-time_series-2020-10-06.zip"

def fetch_opsd_germany_15min():
    """
    Fetch OPSD time series zip, extract the latest CSV,
    parse as pandas dataframe, select German columns, 
    and return clean 15-min resolution dataframe.
    """
    # Download zip
    r = requests.get(OPSD_TIME_SERIES_URL)
    r.raise_for_status()
    
    # Open zip in memory
    with ZipFile(BytesIO(r.content)) as z:
        # OPSD zip has a CSV named like opsd-time_series-YYYY-MM-DD.csv
        csv_name = [f for f in z.namelist() if f.startswith("opsd-time_series") and f.endswith(".csv")][0]
        with z.open(csv_name) as f:
            # parse CSV
            df = pd.read_csv(
                f,
                sep=",",  # OPSD CSV uses commas
                index_col="utc_timestamp",
                parse_dates=True,
                na_values=["NaN", ""]
            )
    
    # Select German columns
    cols = [
        "DE_load_actual_entsoe_transparency",
        "DE_load_forecast_entsoe_transparency",
        "DE_solar_capacity",
        "DE_solar_generation_actual",
        "DE_wind_capacity",
        "DE_wind_generation_actual",
        "DE_wind_onshore_generation_actual",
        "DE_wind_offshore_generation_actual",
    ]
    
    df = df[cols].copy()
    
    # convert numeric columns
    df = df.apply(pd.to_numeric, errors='coerce')
    
    # Optional: enforce 15-min frequency only
    df = df[df.index.to_series().diff().dt.total_seconds().fillna(0) == 900]
    
    return df


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
# Set style
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (20, 6)
# Quick overview: plot load, solar, wind generation
plt.figure()
plt.plot(df_de.index, df_de["DE_load_actual_entsoe_transparency"], label="Load (MW)", color="black")
plt.plot(df_de.index, df_de["DE_solar_generation_actual"], label="Solar (MW)", color="gold")
plt.plot(df_de.index, df_de["DE_wind_generation_actual"], label="Wind (MW)", color="skyblue")
plt.plot(df_de.index, df_de["DE_wind_onshore_generation_actual"], label="Wind Onshore (MW)", color="dodgerblue", alpha=0.6)
plt.plot(df_de.index, df_de["DE_wind_offshore_generation_actual"], label="Wind Offshore (MW)", color="navy", alpha=0.6)
plt.title("German Load and Renewable Generation")
plt.xlabel("Time")
plt.ylabel("Power (MW)")
plt.legend()
plt.show()
# Stacked generation contribution relative to load
plt.figure()
plt.stackplot(
    df_de.index,
    df_de["DE_solar_generation_actual"].fillna(0),
    df_de["DE_wind_onshore_generation_actual"].fillna(0),
    df_de["DE_wind_offshore_generation_actual"].fillna(0),
    labels=["Solar", "Wind Onshore", "Wind Offshore"],
    colors=["gold", "dodgerblue", "navy"],
    alpha=0.8,
)
plt.plot(df_de.index, df_de["DE_load_actual_entsoe_transparency"], label="Load", color="black", linewidth=1.5)
plt.title("Stacked Renewable Generation vs Load")
plt.xlabel("Time")
plt.ylabel("Power (MW)")
plt.legend(loc="upper left")
plt.show()
# Compare capacities vs actual generation
plt.figure()
plt.plot(df_de.index, df_de["DE_solar_capacity"], label="Solar Capacity", color="gold", linestyle="--")
plt.plot(df_de.index, df_de["DE_wind_capacity"], label="Wind Capacity", color="skyblue", linestyle="--")
plt.plot(df_de.index, df_de["DE_solar_generation_actual"].fillna(0), label="Solar Generation", color="orange")
plt.plot(df_de.index, df_de["DE_wind_generation_actual"].fillna(0), label="Wind Generation", color="blue")
plt.title("Installed Capacity vs Actual Generation")
plt.xlabel("Time")
plt.ylabel("Power (MW)")
plt.legend()
plt.show()
# Optional: Zoom into one week to see variability and gaps
one_week = df_de["2015-01-01":"2015-01-07"]
plt.figure()
plt.plot(one_week.index, one_week["DE_load_actual_entsoe_transparency"], label="Load", color="black")
plt.plot(one_week.index, one_week["DE_solar_generation_actual"].fillna(0), label="Solar", color="gold")
plt.plot(one_week.index, one_week["DE_wind_generation_actual"].fillna(0), label="Wind", color="skyblue")
plt.title("One Week Zoom: Load and Renewable Generation")
plt.xlabel("Time")
plt.ylabel("Power (MW)")
plt.legend()
plt.show()

In [ ]:
DATA_DIR = Path("../data/processed")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# ── Load ──────────────────────────────────────────────────────────────────────
load_cols = [c for c in df_de.columns if "load" in c]
df_de[load_cols].to_csv(DATA_DIR / "de_load_15min.csv")

# ── Solar ─────────────────────────────────────────────────────────────────────
solar_cols = [c for c in df_de.columns if "solar" in c]
df_de[solar_cols].to_csv(DATA_DIR / "de_solar_15min.csv")

# ── Wind ──────────────────────────────────────────────────────────────────────
wind_cols = [c for c in df_de.columns if "wind" in c]
df_de[wind_cols].to_csv(DATA_DIR / "de_wind_15min.csv")

# ── Sanity check ──────────────────────────────────────────────────────────────
print("Saved files:")
for f in sorted(DATA_DIR.glob("*.csv")):
    print(f"  {f.name:<35} {f.stat().st_size / 1024:>8.1f} KB")